In [2]:
import pufferlib
from pufferlib.ocean.breakout.breakout import Breakout
import pufferlib.vector
import numpy as np
import torch
import torch.nn as nn 
from stable_baselines3.common.buffers import ReplayBuffer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import time
print(f"Using device: {device}")
from torch.distributions import Categorical
import torch.nn.functional as F

import torch
import gc
import wandb

# Run garbage collector to free Python memory references
gc.collect()

# Empty the PyTorch CUDA cache
torch.cuda.empty_cache()

# Also reset the CUDA memory allocator — clears *reserved* memory
torch.cuda.ipc_collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()


Using device: cuda


In [3]:
class AgentNetwork(nn.Module):
    def __init__(self, vecenv, enc_hidden_size, lstm_hidden_size, lstm_num_layers, device):
        super().__init__()
        self.obs_dim = vecenv.single_observation_space.shape[0]
        self.num_action = vecenv.single_action_space.n
        self.enc_hidden_size = enc_hidden_size
        self.lstm_hidden_size = lstm_hidden_size
        self.lstm_num_layers = lstm_num_layers
        self.num_envs = vecenv.num_agents
        self.device = device
        
        # Define layers
        self.encoder = nn.Linear(self.obs_dim, self.enc_hidden_size).to(self.device)
        self.lstm = nn.LSTM(self.enc_hidden_size, self.lstm_hidden_size, self.lstm_num_layers).to(self.device)
        self.decoder = nn.Linear(self.lstm_hidden_size, self.num_action).to(self.device)
        self.value_head = nn.Linear(self.lstm_hidden_size, 1).to(self.device)

    def init_state(self):
        h0 = torch.zeros(self.lstm_num_layers, self.num_envs, self.lstm_hidden_size, device=self.device)
        c0 = torch.zeros(self.lstm_num_layers, self.num_envs, self.lstm_hidden_size, device=self.device)
        state = (h0, c0)
        return state
    
    def forward(self, obs, state=None): 
        if len(obs.shape) == 2:
            obs = obs.unsqueeze(0)  # Set context length = 1 when just seeing a single batch of observations
        # obs : [context_length, num_envs, obs_dim]
        context_length = obs.shape[0]
        
        emb = torch.relu(self.encoder(obs.view(-1, self.obs_dim)))  # [context_length * num_envs, enc_hidden_size]
        lstm_outs, state = self.lstm(emb.view(context_length, self.num_envs, self.enc_hidden_size), state)
        '''lstm_outs: [context_length, batch_size = num_envs, lstm_hidden_size] 
            state: (h, c) both of shape [num_lstm_layers, batch_size = num_envs, lstm_hidden_size]
        '''
        logits = self.decoder(lstm_outs)  # [context_length, num_envs, num_actions]
        values = self.value_head(lstm_outs).squeeze(-1)  # [context_length, num_envs]
        
        if context_length == 1:
            logits = logits.squeeze(0)  # [num_envs, num_actions]
            values = values.squeeze(0)
            
        return logits,values, state


In [4]:
class ExperienceBuffer:
    def __init__(self, traj_len, agent, device):
        self.traj_len = traj_len
        self.num_envs = agent.num_envs
        self.device = device
        
        self.obs = torch.zeros(traj_len, agent.num_envs, agent.obs_dim, device=device)
        self.old_values = torch.zeros(traj_len, agent.num_envs, device = device)
        self.rewards = torch.zeros(traj_len, agent.num_envs, device=device)
        self.actions = torch.zeros(traj_len, agent.num_envs, device=device)
        self.old_log_probs = torch.zeros(traj_len, agent.num_envs, device=device)
        self.dones = torch.zeros(traj_len, agent.num_envs, device=device)
        self.lstm_h = torch.zeros(traj_len, agent.lstm_num_layers, agent.num_envs, agent.lstm_hidden_size, device=device)
        self.lstm_c = torch.zeros(traj_len, agent.lstm_num_layers, agent.num_envs, agent.lstm_hidden_size, device=device)
        self.value_Tp1  = None

    def yield_slices(self, chunk_size, randomize=False):
        if self.traj_len % chunk_size != 0:
            raise ValueError(f"traj_len ({self.traj_len}) must be divisible by chunk_size ({chunk_size})")
        
        start_list = torch.arange(0, self.traj_len - chunk_size + 1, chunk_size)
        if randomize:
            start_list = start_list[torch.randperm(start_list.size(0))]

        for start in start_list:
            final_V = self.old_values[start + chunk_size] if start < self.traj_len - chunk_size else self.value_Tp1
            
            yield {
                'obs': self.obs[start:start + chunk_size],
                'rewards': self.rewards[start:start + chunk_size],
                'actions': self.actions[start:start + chunk_size],
                'old_log_probs': self.old_log_probs[start:start + chunk_size],
                'dones': self.dones[start:start + chunk_size],
                'lstm_state': (self.lstm_h[start], self.lstm_c[start]),
                'final_V': final_V
            }

    def zero_out(self): 
        self.obs.zero_()
        self.old_values.zero_()
        self.rewards.zero_()
        self.actions.zero_()
        self.old_log_probs.zero_()
        self.dones.zero_()
        self.lstm_h.zero_()
        self.lstm_c.zero_()
        self.value_Tp1 = None  # Or keep it as is if it gets overwritten later

    def get_average_return (self): 
        return self.rewards.sum(dim = 0).mean()

In [8]:
def collect_experience(traj_len, vecenv, agent, device, buffer):
    with torch.no_grad():
        ob, _ = vecenv.reset()
        state = agent.init_state()
        
        for step in range(traj_len):
            ob_tensor = torch.tensor(ob, dtype=torch.float32, device=device)
            buffer.obs[step] = ob_tensor
            buffer.lstm_h[step] = state[0]
            buffer.lstm_c[step] = state[1]
            
            
            logits, values, state = agent(ob_tensor, state)
            dist = Categorical(logits=logits)
            actions = dist.sample()
            log_probs = dist.log_prob(actions)
            
            buffer.old_log_probs[step] = log_probs
            buffer.actions[step] = actions
            buffer.old_values[step] = values
            
            ob, rewards, dones, _, _ = vecenv.step(actions.cpu().numpy())
            buffer.dones[step] = torch.tensor(dones, device=device)
            buffer.rewards[step] = torch.tensor(rewards, device=device)
            state = state * (1-dones)
        
        
        _, final_value, _ = agent(torch.tensor(ob, dtype=torch.float32, device=device), state)
        done_T = buffer.dones[-1]
        buffer.value_Tp1 = final_value * (1.0 - done_T)

def compute_gae(rewards, dones, values, value_Tp1, gamma, lam, context_len):
    
    # Initialize advantage tensor
    A_curr = torch.zeros_like(values[0])  # shape: [num_envs]
    advantages = torch.zeros_like(values)  # shape: [context_len, num_envs]

    for t in reversed(range(context_len)):
        if t == context_len - 1:  # last timestep
            delta = rewards[t] + (gamma * value_Tp1) - values[t]
            A_curr = delta  # No future advantage to consider, just the delta
            advantages[t] = A_curr
        else:  # middle timesteps
            delta = rewards[t] + (gamma * (1.0 - dones[t]) * values[t + 1]) - values[t]
            A_curr = delta + (gamma * lam * advantages[t + 1])
            advantages[t] = A_curr
    
    return advantages

    clipped_objective = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
    
    # The final actor loss is the min of the objective and clipped objective
    loss = -torch.min(objective, clipped_objective).mean()  # Mean over the batch
    return loss

def critic_loss (values, returns):
    #critic loss is just mean squared error between predicited values by the critic and return estimates. 
    return F.mse_loss(values,returns)



In [9]:
def train_epoch(epoch_number, total_epochs, vecenv, agent: AgentNetwork, buffer: ExperienceBuffer,
                traj_len, context_len, random_slices, optimizer, lr, epsilon, gamma, lam,
                critic_coeff, entropy_coeff, device):

    buffer.zero_out()
    collect_experience(traj_len, vecenv, agent, device, buffer)

    avg_return = buffer.get_average_return()
    print(f"average_trajectory_return: {avg_return}, epoch: {epoch_number}/{total_epochs}")
   

    total_batches = traj_len // context_len
    wandb.log({"average_return": avg_return}, step=epoch_number*total_batches)

    for i, batch in enumerate(buffer.yield_slices(context_len, random_slices)):
        global_step = total_batches * epoch_number + i + 1
        init_state = tuple(v.detach() for v in batch["lstm_state"])
        logits = torch.zeros(context_len,agent.num_envs, agent.num_action).to(device)
        values = torch.zeros(context_len, agent.num_envs).to(device)
        state = init_state
        for step in range(context_len):
            state = state * (1-(batch["dones"][step]))
            logit, value, new_state = agent(batch['obs'][step],state)
            logits[step] = logit
            values[step] = value
            state = new_state
            
        dist = Categorical(logits=logits)
        new_log_probs = dist.log_prob(batch['actions'])
        entropies = dist.entropy()
        advantages = compute_gae(batch['rewards'], batch['dones'], values, batch['final_V'],
                                 gamma, lam, context_len)
        returns = values + advantages

        # Individual losses
        actorloss = actor_loss(batch['old_log_probs'], new_log_probs, advantages, epsilon)
        criticloss = critic_coeff *critic_loss(values, returns)
        entropy_bonus = entropy_coeff * entropies.mean()

        # Total loss
        loss = actorloss +  criticloss - entropy_bonus

        # Backprop + optimizer
        optimizer.zero_grad()
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(agent.parameters(), max_norm=0.5)
        optimizer.step()

        # Clip fraction and ratio under no_grad
        with torch.no_grad():
            ratio = torch.exp(new_log_probs - batch['old_log_probs'])
            clip_fraction = (torch.abs(ratio - 1.0) > epsilon).float().mean()

        # Logging
        wandb.log({
            "loss/total": loss.item(),
            "loss/policy": actorloss.item(),
            "loss/value": criticloss.item(),
            "loss/entropy_bonus": entropy_bonus.item(),
            "entropy": entropies.mean().item(),
            "clip_fraction": clip_fraction.item(),
            "advantage_mean": advantages.mean().item(),
            "advantage_std": advantages.std().item(),
            "advantage_max": advantages.max().item(),
            "advantage_min": advantages.min().item(),
            "grad_norm": grad_norm.item(),
            "lr": optimizer.param_groups[0]['lr']
        }, step=global_step)

        print(f"mini_batch: {i+1}/{total_batches}, total_loss: {loss.item():.4f}")




In [10]:

traj_len = 4096
context_len = 16
num_envs = 1024
obs_dim = 118
gamma = 0.99
lam = 0.95
epsilon = 0.2  # Clipping epsilon for PPO
critic_coeff = 0.5  # Weight for the critic loss
entropy_coeff = 0.01
lr = 1e-4
total_epochs = 10
random_slices = True

# Environment settings
env_kwargs = {'num_envs': num_envs}
train_kwargs = {'num_envs': 1, 'num_workers': 1, 'env_batch_size': 1}

# Agent network settings
enc_hidden_size = 128
lstm_hidden_size = 128
lstm_num_layers = 1

# Create environment
make_env = Breakout  # Assuming Breakout is defined elsewhere
vecenv = pufferlib.vector.make(
    make_env,
    env_kwargs=env_kwargs,
    num_envs=train_kwargs['num_envs'],
    num_workers=train_kwargs['num_workers'],
    batch_size=train_kwargs['env_batch_size'],
    backend=pufferlib.vector.Multiprocessing
)

# Initialize agent

agent = AgentNetwork(vecenv, enc_hidden_size, lstm_hidden_size, lstm_num_layers, device)
buffer = ExperienceBuffer(traj_len, agent, device)
# Initialize optimizer
optimizer = torch.optim.Adam(agent.parameters(), lr=lr)
wandb.init(
    project="ppo-training",
    name=f"ppo-run",
    config={
        "lr": lr,
        "gamma": gamma,
        "lam": lam,
        "epsilon": epsilon,
        "critic_coeff": critic_coeff,
        "entropy_coeff": entropy_coeff,
        "traj_len": traj_len,
        "context_len": context_len
    }
)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


In [11]:
for epoch_number in range(1, total_epochs + 1): 
    train_epoch(epoch_number, total_epochs, vecenv, agent, buffer,
                traj_len, context_len, random_slices, optimizer, lr, epsilon, gamma, lam,
                critic_coeff, entropy_coeff, device)  
    
wandb.finish()

TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.